In [1]:
# matrix_calculations.py ---
#
# Filename: matrix_calculations.py
# Description: This code make the calculation of the connectivity matrix for the GBR
# Author: Javier Porobic
# Maintainer: Javier Porobic concat "javier.porobicgarate" "@" "csiro" ".au")
# Created: Thu Feb 16 13:05:47 2023 (+1100)
# Version: 0.0.1
# Package-Requires: ()
# Last-Updated:
#           By:Javier Porobic
#     Update #: 29
# URL:
# Doc URL:
# Keywords:
# Compatibility:
# This piece of code has been created to work on standard versions of python with
# publicly available libraries. The only configuration that has been hardwired is the
# location of the files and the commands related to the work on the high-performance
# computer of CSIRO.
#

# Commentary:
# This code utilizes particle dispersal tracks generated by OceanParcels to evaluate
# the level of connectivity among the Great Barrier Reef (GBR) reefs. The
# calculations for total connectivity take into account decay and competence, and are
# based on the functions established by Moneghetti et al. in 2019. These functions
# allow for a comprehensive assessment of the dispersal dynamics of particles,
# providing insights into the connectivity patterns among the GBR reefs. The code
# serves as a valuable tool for understanding the interconnectivity of reef systems
# and can contribute to informed decision-making in reef management and conservation
# efforts.
#

# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation, either version 3 of the License, or (at
# your option) any later version.
#
# This program is distributed in the hope that it will be useful, but
# WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU
# General Public License for more details.
import xarray as xr
from scipy.integrate import quad
import geopandas as gpd
import numpy as np
import sys
from shapely.geometry import Point
import shapely
import math
from tqdm import tqdm
import pandas as pd
from numba import jit, njit
# Numba translates Python functions to optimized machine code at runtime.
import numba
import os
from joblib import Parallel, delayed, parallel_backend
import time

## ~~~~~~~~~~~~~~~~~~ ##
## ~     Functions  ~ ##
## ~~~~~~~~~~~~~~~~~~ ##


def bathtub_curve(lmbda, v, sigma):
    """
    Transforms the bathtub equation into a function that can be integrated
    using the trapezoidal function.

    Parameters
    ----------
    lmbda : float
        The scale parameter of the Weibull distribution used in the bathtub equation.
        Must be a positive integer or float.
    v : float
        The shape parameter of the Weibull distribution used in the bathtub equation.
        Must be a positive integer or float.
    sigma : float
        A constant that determines the shape of the bathtub curve. Must be a positive
        integer or float. It can be zero, becoming an exponential function.

    Returns
    -------
    function
        A lambda function that represents the bathtub curve equation. The function takes
        a single argument, t, which represents the age of the coral. The function returns
        the value of the bathtub curve equation at that age.
    """

    def u(t): return (lmbda * v * pow((lmbda * t), v - 1)) / \
        (1 - sigma * pow((lmbda * t), v))
    return (u)


def piecewise_decay(ages, Tcp, lmbda1, lmbda2, v1, v2, sigma1, sigma2):
    """
    Calculates the probability of survival of corals larvaes at different ages, using the piecewise
    Weibull-Weibull survival model described by Moneghetti et al. (2019).

    Parameters
    ----------
    ages : list
        A list of ages of the corals. Each age must be a positive integer or float.
    Tcp : float
        The age (inflection point) at which the corals transition from a Weibull survival curve to another Weibull
        survival curve. Must be greater than 0 and less than the maximum age in `ages`.
    lmbda1 : float
        The scale parameter of the Weibull survival curve in the first phase. Must be a positive
        integer or float.
    lmbda2 : float
        The scale parameter of the Weibull survival curve in the second phase. Must be a positive
        integer or float.
    v1 : float
        The shape parameter of the Weibull survival curve in the first phase. Must be a positive
        integer or float.
    v2 : float
        The shape parameter of the Weibull survival curve in the second phase. Must be a positive
        integer or float.
    sigma1 : float
        The standard deviation of the Gaussian noise added to the survival curve in the first phase.
        Must be a positive integer or float. It can be zero, becoming an exponential function.
    sigma2 : float
        The standard deviation of the Gaussian noise added to the survival curve in the second phase.
        Must be a positive integer or float. It can be zero, becoming an exponential function.

    Returns
    -------
    list
        A list of survival probabilities for the corals, calculated using the piecewise
        Weibull-Weibull survival model. Each survival probability corresponds to the age
        in the input list `ages`.
    """
    fx1 = bathtub_curve(lmbda1, v1, sigma1)
    fx2 = bathtub_curve(lmbda2, v2, sigma2)
    decay = []
    for age in range(0, len(ages)):
        if (ages[age] < Tcp):
            area = quad(fx1, 0, ages[age])[0]
        else:
            area = quad(fx1, 0, Tcp)[0] + quad(fx2, Tcp, ages[age])[0]
        decay.append(math.exp(-area))
    return decay


def exponential_growth_function(initial_value, growth_rate, time_period):
    """
    Calculates the exponential growth over a given time period.

    Parameters:
    - initial_value (float): The starting value of the exponential growth.
    - growth_rate (float): The rate at which the value is growing, expressed as a decimal or fraction.
    - time_period (int): The number of time periods over which the growth occurs (larval age in days).

    Returns:
    - float: The value after the exponential growth.
    """
    return initial_value * pow((1 + growth_rate), time_period)


def exponential_decay_function(initial_value, decay_rate, time, initial_time=0.0):
    """
    Calculate the value of an exponentially decaying quantity at a given time.
    This is a very simple decay function assuming mean lifetime decay for the entire particle
    Parameters:
    - initial_value: float
        The initial value of the decaying quantity (this is set to 1 as is a proportion).
    - decay_rate: float
        The rate at which the quantity decays per unit time.
    - time: float
        The time at which the decay is evaluated (age of the larvae).
    Returns:
    - result: float
       The value of the decaying quantity at the given time.
    """
    return initial_value * np.exp(-decay_rate * (time - initial_time))


def exponential_competence(ages, tc, peak_age):
    """
    Calculates the competence level based on age using an exponential growth function.

    Parameters:
    - ages (list): A list of ages.
    - tc (float): The threshold age below which the competence level is 0.
    - peak_age (float): The age at which the competence level reaches its peak (1).

    Returns:
    - list: A list of competence levels corresponding to each age.
    """
    competence = []
    for age in range(0, len(ages)):
        if (ages[age] < tc):
            probability = 0
        elif (ages[age] < peak_age):
            probability = exponential_growth_function(0.01, 0.245, ages[age])
        else:
            probability = 1
        competence.append(probability)
    return competence


def exponential_decay(ages, tc, phases_age):
    """
    Calculates the decay probability based on age using an exponential decay function.

    Parameters:
    - ages (list): A list of ages.
    - tc (float): The threshold age below which the decay probability is 0.
    - phases_age (float): The age at which the decay probability transitions to a different rate.

    Returns:
    - list: A list of decay probabilities corresponding to each age.

    """
    decay = []
    for age in range(0, len(ages)):
        if (ages[age] < tc):
            probability = 0
        elif (ages[age] < phases_age):
            probability = exponential_decay_function(1.0, 0.2, ages[age])
        else:
            probability = exponential_decay_function(1.0, 0.5, ages[age])
        decay.append(probability)
    return decay


def piecewise_competence(ages, tc, Tcp, alpha, beta1, beta2, v):
    """
    Calculates the larval competence values at different ages (days), using the piecewise
    Weibull-exponential competence model. This function is a replica of the R code used by
    Moneghetti et al. (2019) to calculate competence.

    Parameters
    ----------
    ages : list
        A list of larvaes ages. Each age must be a positive integer or float.
    tc : float
        The age at which the larvaes reaches their maximum competence level. Must be a
        positive integer or float.
    Tcp : float
        The age at which the larvaes starts to experience a decline in competence. Must be
        greater than tc and a positive integer or float.
    alpha : float
        The scale parameter of the Weibull distribution. Must be a positive integer or float.
    beta1 : float
        The shape parameter of the Weibull distribution in the early decline phase. Must be
        a positive integer or float.
    beta2 : float
        The shape parameter of the Weibull distribution in the late decline phase. Must be
        a positive integer or float.
    v : float
        The exponential decay parameter in the early decline phase. Must be a positive
        integer or float.

    Returns
    -------
    list
        A list of competence values for larvaes, calculated using the piecewise
        Weibull-exponential competence model. Each competence value corresponds to the age
        in the input list `ages`.
    """
    competence = []
    for age in range(0, len(ages)):
        if (ages[age] < tc):
            area = 0
        if (ages[age] >= tc and ages[age] <= Tcp):
            def fxtau_early(tau): return alpha * math.exp(-alpha *
                                                          (tau - tc)) * math.exp(- pow((beta1 * (ages[age]-tau)), v))
            area = quad(fxtau_early, tc, ages[age])[0]
        if (ages[age] > Tcp):
            def fxtau_late_first(tau): return alpha * math.exp(-alpha * (tau - tc)) * \
                math.exp(- pow((beta1 * (Tcp-tau)), v)) * \
                math.exp(-beta2 * (ages[age] - Tcp))
            def fxtau_late_second(tau): return alpha * math.exp(-alpha *
                                                                (tau - tc)) * math.exp(- beta2 * (ages[age]-tau))
            area = quad(fxtau_late_first, tc, Tcp)[
                0] + quad(fxtau_late_second, Tcp, ages[age])[0]
        competence.append(area)
    return (competence)



In [2]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ~~~ In polygon algorithm and optimizers
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


@jit(nopython=True)
def point_in_polygon(x, y, polygon):
    """
    Determines whether a point is inside a polygon using the ray-casting algorithm.

    Parameters
    ----------
    x : float
        The x-coordinate of the point to test.
    y : float
        The y-coordinate of the point to test.
    polygon : list of tuples
        A list of tuples representing the vertices of the polygon, in order.

    Returns
    -------
    bool
        True if the point is inside the polygon, False otherwise.
    """
    num_vertices = len(polygon)
    is_inside = False
    previous_x, previous_y = polygon[0]
    # start outside the polygon
    intersection_x = 0.0
    for i in numba.prange(num_vertices + 1):
        current_x, current_y = polygon[i % num_vertices]
        if y > min(previous_y, current_y):
            if y <= max(previous_y, current_y):
                if x <= max(previous_x, current_x):
                    if previous_y != current_y:
                        intersection_x = (
                            y - previous_y) * (current_x - previous_x) / (current_y - previous_y) + previous_x
                    if previous_x == current_x or x <= intersection_x:
                        is_inside = not is_inside
        previous_x, previous_y = current_x, current_y

    return is_inside


@njit(parallel=False)
def points_in_polygon(xs, ys, miny, maxy, polygon):
    """
    Test whether a point is inside a given polygon using the point-in-polygon algorithm.

    This function tests each point in the `points` array to determine if it is inside the polygon
    defined by the vertices in the `polygon` array. The function uses the point-in-polygon Ray casting
    algorithm to perform this test. The algorithm performs the even-odd-rule algorithm to find out
    whether a point is in a given polygon. This runs in O(n) time where n is the number of edges of the polygon.

    Parameters:
    -----------
    points: numpy.ndarray of shape (n, 2)
        Array of n points to test. Each point is defined by its x and y coordinates in columns 0 and 1 respectively.
    polygon: numpy.ndarray of shape (m, 2)
        Array of m vertices defining the polygon. Each vertex is defined by its x and y coordinates in columns 0 and 1 respectively.

    Returns:
    --------
    D: numpy.ndarray of shape (n,)
        Boolean array indicating whether each point in `points` is inside the polygon (`True`) or not (`False`).
    """
    D = np.empty(len(ys), dtype=numba.boolean)
    for i in range(len(D)):
        if ys[i] >= miny and ys[i] <= maxy:
            D[i] = point_in_polygon(xs[i], ys[i], polygon)
        else:
            D[i] = False
    return D




In [55]:
### Main Matrix Calculations
def calc(path, release_start_day, source_reef, num_reefs, data_shape, tc_CoTS, peak_age, phase_age):
    """
    Calculates the connectivity metrics for a given source reef using particle dispersal tracks.

    Args:
      source_reef (int): The index of the source reef to calculate connectivity from.

    Returns:
      list: A list containing four numpy arrays representing the connectivity metrics:
            - connectivity_matrix_max: Maximum connectivity values for each reef in the study area.
            - connectivity_matrix_sum: Sum of connectivity values for each reef in the study area.
            - connectivity_variance_max: Variance of maximum connectivity values for each reef.
            - connectivity_variance_sum: Variance of sum of connectivity values for each reef.
    """
    t = time.time()
    # Creating empty arrays
    connectivity_matrix_sum = np.zeros(num_reefs)
    connectivity_matrix_max = np.zeros(num_reefs)
    connectivity_variance_sum = np.zeros(num_reefs)
    connectivity_variance_max = np.zeros(num_reefs)

    file_name = path + "/GBR1_H2p0_CoTs_Release_" + release_start_day + \
        "_Polygon_" + str(source_reef) + '_No_Wind_drift_displacement_field.nc'
    if not os.path.exists(file_name):
        Warning('file missing - ' + str(source_reef))
        pass
    else:
        print('start simulation reefs: ', source_reef)
        output_nc = xr.open_dataset(file_name)
        #print(output_nc.variables)
        ntraj = output_nc.dims['traj']

        particles = pd.DataFrame({
            'latitudes': output_nc['lat'].values.ravel(),
            'longitudes': output_nc['lon'].values.ravel(),
            'trajectories': output_nc['trajectory'].values.ravel(),
            'age': output_nc['age'].values.ravel() / 86400  # Seconds to days
        })
        output_nc.close()
        print('Step_1 reading netcdf: ')
        # Cleaning the nans
        particles = particles.dropna()
        # remove particles bellow minimum age and maximum age
        particles = particles[particles['age'] >= tc_CoTS]
        print('Step_2 Particle age: ')
        # set particles boundaries in model domain
        # this avoids the overload of looking over all the reefs
        if(np.isnan(particles['latitudes'].values).all()):
            Warning('The latitude array is full of nans\n there is no connection to this reef')
        else:
            particle_max_lat = np.nanmax(particles['latitudes'].values)
            print(particle_max_lat)
            particle_min_lat = np.nanmin(particles['latitudes'].values)
            print('Step_3 Lat and lon Readed')
            # making boolean series
            upper_bound = data_shape['min_lat'] <= particle_max_lat
            mmax = upper_bound.to_numpy()
            inf_bound = data_shape['max_lat'] >= particle_min_lat
            minf = inf_bound.to_numpy()
            boundary_reefs = np.where(np.multiply(minf, mmax))[0]

            print('Step_4 entering the main loop')
            
            for sink_reef in boundary_reefs:
                reef_index = list(
                    data_shape.loc[data_shape['FID'] == sink_reef].index)
                reef_index = int("".join(map(str, reef_index)))
                polygon = np.array(
                    list(data_shape['geometry'][sink_reef].exterior.coords))
                if (particles.size == 0):
                    break  # breaking the loop if not more particles
                # Are these points inside the polygon?
                p = points_in_polygon(
                    particles['longitudes'].values,
                    particles['latitudes'].values,
                    data_shape.min_lat[sink_reef],
                    data_shape.max_lat[sink_reef],
                    polygon,
                )
                m = np.where(p)[0]
                if (len(m > 0)):
                    settled_age = particles['age'].iloc[m].values
                    settled_traj = particles['trajectories'].iloc[m].values
                    decay = exponential_decay(settled_age, tc_CoTS, peak_age)
                    competence = exponential_competence(
                        settled_age, tc_CoTS, phase_age)
                    connect = pd.DataFrame({'connect': np.array(
                        decay) * np.array(competence), 'traj': settled_traj})
                    connect_sum = (connect.groupby('traj')[
                                'connect'].sum()/ntraj).to_numpy()
                    connect_max = (connect.groupby('traj')[
                                'connect'].max()/ntraj).to_numpy()
                    # Average connectivity through all the particles in that reef
                    connectivity_matrix_max[reef_index] = connect_max.mean()
                    connectivity_matrix_sum[reef_index] = connect_sum.mean()
                    # Variance
                    connectivity_variance_max[reef_index] = connect_max.var()
                    connectivity_variance_sum[reef_index] = connect_sum.var()
                    particles.drop(index=particles.iloc[m].index, inplace=True)

        print('Done', source_reef, int(time.time() - t + 0.5), flush=True)

    return [connectivity_matrix_max, connectivity_matrix_sum, connectivity_variance_max, connectivity_variance_sum]




In [56]:
## ~~~~~~~~~~~~~~~~~~~~ ##
## ~      Parameters  ~ ##
## ~~~~~~~~~~~~~~~~~~~~ ##
# Parameters used in the decay and competence function
# Decay
Tcp_decay = 2.583
lmbda1 = 0.4
lmbda2 = 0.019
v1 = 2.892
v2 = 1.716
sigma1 = 0
sigma2 = 0

# Competence for corals
tc = 14
Tcp_comp = 69.91245
alpha = 1.295
beta1 = 0.001878001
beta2 = 0.3968972
v = 0.364

# Exponential decay for CoT
tc_CoTS = 14.0
phase_age = 21.0
peak_age = 21.0




In [58]:
## ~~~~~~~~~~~~~~~ ##
## ~   Main Code ~ ##
## ~~~~~~~~~~~~~~~ ##
shapefile = '/home/por07g/Documents/Projects/GBR_modeling/oceanparcels_cots/Shape_files/gbr1_cots_2m_merged_buffer0p005.shp'
#shapefile = '/datasets/work/oa-coconet/work/oceanparcels_gbr_CoTs/Shape_files/gbr1_cots_2m_merged_buffer0p005.shp'
data_shape = gpd.read_file(shapefile)

# getting the boundaries of each reefs for
# entired GBR
num_reefs = data_shape.shape[0]
min_lat = []
max_lat = []
for i_polygon in range(0, num_reefs):
    min_lat.append(
        np.nanmin(np.array(data_shape['geometry'][i_polygon].bounds)[[1, 3]]))
    max_lat.append(
        np.nanmax(np.array(data_shape['geometry'][i_polygon].bounds)[[1, 3]]))
data_shape['min_lat'] = min_lat
data_shape['max_lat'] = max_lat

#release_start_day = sys.argv[1]
release_start_day = "2019-12-15"
#/home/por07g/Documents/Projects/GBR_modeling/Connectivity/Debug_codes/outputs/2015-12-01
path='/home/por07g/Documents/Projects/GBR_modeling/Connectivity/Debug_codes/outputs/' + release_start_day

#path = '/datasets/work/oa-coconet/work/OceanParcels_outputs/CoTs/' + release_start_day
outputpath ='/home/por07g/Documents/Projects/GBR_modeling/Connectivity/Debug_codes/outputs/'
#outputpath = '/datasets/work/oa-coconet/work/OceanParcels_outputs/CoTs/Matrices/'
jobs = range(num_reefs)
#n_jobs = int(os.getenv('SLURM_CPUS_ON_NODE', 10))
n_jobs = 1
with parallel_backend(backend='loky', n_jobs=n_jobs):
    results_list = Parallel()(delayed(calc)(path, release_start_day, source_reef, num_reefs, data_shape, tc_CoTS, peak_age, phase_age) for source_reef in jobs)

print('calculations done', time.strftime("%H:%M:%S"), flush=True)

# Creating empty arrays
connectivity_matrix_sum = np.empty((num_reefs, num_reefs))
connectivity_matrix_max = np.empty((num_reefs, num_reefs))
connectivity_variance_sum = np.zeros((num_reefs, num_reefs))
connectivity_variance_max = np.zeros((num_reefs, num_reefs))

for k in jobs:
    connectivity_matrix_max[k, :] = results_list[k][0]
    connectivity_matrix_sum[k, :] = results_list[k][1]
    connectivity_variance_max[k, :] = results_list[k][2]
    connectivity_variance_sum[k, :] = results_list[k][3]

np.savetxt(outputpath + release_start_day +
           "_Connectivity_sum_CoTS.csv", connectivity_matrix_sum, delimiter=",")
np.savetxt(outputpath + release_start_day +
           "_Connectivity_max_CoTS.csv", connectivity_matrix_max, delimiter=",")
np.savetxt(outputpath + release_start_day + "_Variance_max_CoTS.csv",
           connectivity_variance_max, delimiter=",")
np.savetxt(outputpath + release_start_day + "_Variance_sum_CoTS.csv",
           connectivity_variance_sum, delimiter=",")
# matrix_calculations.py ends here



start simulation reefs:  3446
Step_1 reading netcdf: 
Step_2 Particle age: 
-10.728177
Step_3 Lat and lon Readed
Step_4 entering the main loop
Done 3446 0
start simulation reefs:  3447
Step_1 reading netcdf: 
Step_2 Particle age: 
Done 3447 0
start simulation reefs:  3448
Step_1 reading netcdf: 
Step_2 Particle age: 
-10.732726
Step_3 Lat and lon Readed
Step_4 entering the main loop
Done 3448 0
start simulation reefs:  3449
Step_1 reading netcdf: 
Step_2 Particle age: 
-9.080924
Step_3 Lat and lon Readed
Step_4 entering the main loop
Done 3449 1
start simulation reefs:  3450
Step_1 reading netcdf: 
Step_2 Particle age: 
-8.092111
Step_3 Lat and lon Readed
Step_4 entering the main loop
Done 3450 0
start simulation reefs:  3451
Step_1 reading netcdf: 
Step_2 Particle age: 
-9.007542
Step_3 Lat and lon Readed
Step_4 entering the main loop
Done 3451 1
start simulation reefs:  3452
Step_1 reading netcdf: 
Step_2 Particle age: 
-9.091185
Step_3 Lat and lon Readed
Step_4 entering the main loo